# Trip-time descriptive statistics

Descriptive statistics for observed trip times along a single shape, across every service date in the by-date vehicle-position dataset. This is a precursor to the delay breakdown (MIT thesis §3.4.5): characterizing the distribution of running times and splitting day vs night trips (night ≈ free-flow baseline) before decomposing delay into components.

A *trip* here is one `TRIP_KEY` within one service date; *trip time* is the observed span of its GPS pings (`max - min` of `event_time_datetime`).

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from _data_loaders import get_culver_city_vehicle_positions, list_available_service_dates

## Configuration

`SHAPE_KEY` selects the route/shape (start with `"105"`). `NIGHT_START_HOUR` is the local hour (24h) at/after which a trip's *start* counts as night. `MIN_TRIP_MINUTES` drops degenerate trips (GPS fragments / deadheads) from the stats and histograms; set to `0` to keep everything.

In [ ]:
SHAPE_KEY = "105"
NIGHT_START_HOUR = 20  # trips starting at/after 22:00 are "night"
MIN_TRIP_MINUTES = 5   # exclude shorter trips as fragments/deadheads

## Load all service dates

Read each per-date geoparquet for the selected shape and stack them, tagging every ping with its `service_date`.

In [ ]:
service_dates = list_available_service_dates()
print(f"{len(service_dates)} service dates: {service_dates[0]} … {service_dates[-1]}")

per_date_positions = [
    get_culver_city_vehicle_positions([SHAPE_KEY], service_date).assign(service_date=service_date)
    for service_date in service_dates
]
vp_all = pd.concat(per_date_positions, ignore_index=True)
print(f"{len(vp_all):,} vehicle-position rows for shape {SHAPE_KEY}")

## Per-trip table

Collapse pings to one row per (`service_date`, `TRIP_KEY`): start, end, duration, ping count, and the day/night period from the start time.

In [ ]:
trip_table = (
    vp_all.groupby(["service_date", "TRIP_KEY"], dropna=True)
    .agg(
        trip_start=("event_time_datetime", "min"),
        trip_end=("event_time_datetime", "max"),
        n_pings=("event_time_datetime", "size"),
    )
    .reset_index()
)

trip_table["duration_min"] = (
    trip_table["trip_end"] - trip_table["trip_start"]
).dt.total_seconds() / 60
trip_table["start_hour"] = trip_table["trip_start"].dt.hour
trip_table["period"] = np.where(
    trip_table["start_hour"] >= NIGHT_START_HOUR, "night", "day"
)

print(f"{len(trip_table)} trips total")
trip_table.head()

## Data quality: short trips

Degenerate trips (very short observed spans) are likely GPS fragments or deadheads rather than full revenue trips. Report them, then keep only trips at least `MIN_TRIP_MINUTES` long for the stats below.

In [ ]:
short_trips = trip_table[trip_table["duration_min"] < MIN_TRIP_MINUTES]
print(f"{len(short_trips)} of {len(trip_table)} trips shorter than {MIN_TRIP_MINUTES} min (excluded)")

trips = trip_table[trip_table["duration_min"] >= MIN_TRIP_MINUTES].copy()
print(f"{len(trips)} trips kept")

## Number of trips per period

In [ ]:
print("Trips per period (all dates):")
print(trips["period"].value_counts())

print("\nTrips per period per service date:")
trips.groupby(["service_date", "period"]).size().unstack(fill_value=0)

## Trip-time descriptive statistics by period

In [ ]:
trips.groupby("period")["duration_min"].describe()

## Histogram of trip times: day vs night

Shared bins so the two periods are directly comparable. Night trips are sparse (few late departures per day), so counts differ substantially.

In [ ]:
bins = np.histogram_bin_edges(trips["duration_min"], bins=30)

fig, ax = plt.subplots(figsize=(10, 5))
for period, color in [("day", "tab:blue"), ("night", "tab:orange")]:
    durations = trips.loc[trips["period"] == period, "duration_min"]
    ax.hist(durations, bins=bins, alpha=0.5, color=color, label=f"{period} (n={len(durations)})")

ax.set_xlabel("Trip duration (minutes)")
ax.set_ylabel("Number of trips")
ax.set_title(f"Shape {SHAPE_KEY}: trip-time distribution, day vs night")
ax.legend()
plt.show()